# Build `ouroboros-cache` Kaggle Dataset

Run this notebook once on Kaggle with internet enabled. It will:

1. Download the full Hugging Face cache for `ai21labs/AI21-Jamba-Reasoning-3B`
2. Build the T4 (`sm75`) wheels for `causal_conv1d` and `mamba_ssm`
3. Assemble `ouroboros_cache/` so it can be published as a Kaggle dataset input

Expected dataset layout after the final cell:

```
/kaggle/working/ouroboros_cache/
  hf_cache/
  wheels/
```

Before running:
- Add a Kaggle secret named `HF_TOKEN`
- Enable internet for the notebook
- Use a T4 session so the built wheels match `sm75`


In [ ]:
import os
from pathlib import Path

os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Missing HF_TOKEN. Add it as a Kaggle secret before running this notebook.')

print(f"HF_HOME={os.environ['HF_HOME']}")


## 1. Download the model into the Hugging Face cache

This uses `snapshot_download` to populate the cache in a reusable HF layout.


In [ ]:
from huggingface_hub import snapshot_download

target_dir = '/kaggle/working/hf_cache/hub/models--ai21labs--AI21-Jamba-Reasoning-3B'
snapshot_download(
    repo_id='ai21labs/AI21-Jamba-Reasoning-3B',
    token=os.environ['HF_TOKEN'],
    local_dir=target_dir,
    local_dir_use_symlinks=False,
)
print('Model download complete.')
print('Hub contents:', os.listdir('/kaggle/working/hf_cache/hub'))


## Optional verification load

Uncomment and run this cell only if you want to verify the cache by doing a real model load. It is not required for dataset assembly.


In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#
# model = AutoModelForCausalLM.from_pretrained(
#     'ai21labs/AI21-Jamba-Reasoning-3B',
#     quantization_config=BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type='nf4',
#         bnb_4bit_compute_dtype=torch.float16,
#         bnb_4bit_use_double_quant=True,
#     ),
#     device_map={'': 0},
#     trust_remote_code=True,
#     low_cpu_mem_usage=True,
# )
# print('Cache populated:', os.listdir('/kaggle/working/hf_cache/hub'))
# del model
# torch.cuda.empty_cache()


## 2. Build the T4 wheels (`sm75`)

This matches the bootstrap logic used by the training notebook, but collects the wheels into one persistent dataset.


In [ ]:
import subprocess
import sys
from pathlib import Path

WHEEL_BASES = [
    ('causal_conv1d', 'causal-conv1d==1.6.1'),
    ('mamba_ssm', 'git+https://github.com/state-spaces/mamba.git@v1.2.2'),
]
WHEEL_OUT = Path('/kaggle/working/wheels')
WHEEL_OUT.mkdir(exist_ok=True)

env = {**os.environ, 'TORCH_CUDA_ARCH_LIST': '7.5+PTX', 'MAX_JOBS': '4'}

for pkg_name, pip_spec in WHEEL_BASES:
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'wheel', pip_spec,
            '--no-build-isolation', '--no-deps', '-w', str(WHEEL_OUT),
        ],
        env=env,
        check=True,
    )
    whl = sorted(WHEEL_OUT.glob(f'{pkg_name}*.whl'))[-1]
    print(f'Built: {whl.name}')

print('Final wheel set:')
for path in sorted(WHEEL_OUT.glob('*.whl')):
    print(' -', path.name)


## 3. Assemble the Kaggle dataset directory


In [ ]:
import shutil
from pathlib import Path

out = Path('/kaggle/working/ouroboros_cache')
(out / 'wheels').mkdir(parents=True, exist_ok=True)

for whl in Path('/kaggle/working/wheels').glob('*.whl'):
    shutil.copy2(whl, out / 'wheels' / whl.name)

shutil.copytree(
    '/kaggle/working/hf_cache',
    str(out / 'hf_cache'),
    dirs_exist_ok=True,
)

print('Cache assembled at:', out)
for path in sorted(out.rglob('*')):
    print(path)


## 4. Publish it as a Kaggle dataset

In the notebook Output tab:
1. Click **New dataset**
2. Name it `ouroboros-cache`
3. Make it **Private** or **Unlisted** as needed
4. Publish the `ouroboros_cache/` directory

Expected slug for Account A:
- `weirdrunner/ouroboros-cache`

Then attach it to `kaggle-utils.ipynb` as an input so it mounts at `/kaggle/input/ouroboros-cache/`.
